# Desafío Final: Modelo predictivo en Apache Spark MLlib
**Descripción:** Construcción de un modelo de clasificación binaria para anticipar transacciones de ventas anómalas (riesgosas) utilizando el entorno distribuido de Spark MLlib.

In [30]:
# Instalación de dependencias necesarias para Google Colab
!pip install pyspark pandas openpyxl

# Desafío Final: Modelo predictivo en Apache Spark MLlib
**Descripción:** Construcción de un modelo de clasificación binaria para anticipar transacciones de ventas anómalas (riesgosas) utilizando el entorno distribuido de Spark MLlib.

In [31]:
# Importación de librerías y creación de la sesión de Spark
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, hour, when
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

spark = SparkSession.builder \
    .appName("Modelo_Predictivo_Riesgos") \
    .getOrCreate()

print("Sesión de Spark inicializada.")

Sesión de Spark inicializada.


## 1. Preparación del Dataset
Carga de datos simulados de ventas, limpieza distribuida y creación de la variable objetivo (`label`). La etiqueta de riesgo se fundamenta en reglas de negocio: compras nocturnas y por montos elevados.

In [32]:
# Carga del Excel a través de Pandas
file_path = 'Apoyo prueba - ventas_simuladas.xlsx'
pdf = pd.read_excel(file_path)

# Forzar conversión numérica
pdf['Monto_Total'] = pd.to_numeric(pdf['Monto_Total'], errors='coerce')
pdf['Precio_Unitario'] = pd.to_numeric(pdf['Precio_Unitario'], errors='coerce')

# Se convierte la fecha con manejo de errores para descartar registros con formato inválido
pdf['Fecha_Hora'] = pd.to_datetime(pdf['Fecha_Hora'], errors='coerce')

# Eliminar nulos críticos
pdf = pdf.dropna(subset=['Fecha_Hora', 'Monto_Total'])

# Ingesta en Spark
df_raw = spark.createDataFrame(pdf)

# A. Limpieza de datos distribuida y extracción temporal
df_clean = df_raw.withColumn("Hora", hour(col("Fecha_Hora")))
df_clean = df_clean.fillna({"Cantidad": 1.0, "Sucursal": "Desconocida", "Producto": "Desconocido"})

# B. Creación del 'label' (Lógica: Monto > 5000 en horario nocturno/madrugada)
df_model = df_clean.withColumn(
    "label",
    when((col("Monto_Total") > 5000) & ((col("Hora") < 6) | (col("Hora") > 22)), 1).otherwise(0)
)

# C. Preprocesamiento: Indexación y Vectorización (Prevención de Data Leakage)
sucursal_indexer = StringIndexer(inputCol="Sucursal", outputCol="Sucursal_Index", handleInvalid="keep")
producto_indexer = StringIndexer(inputCol="Producto", outputCol="Producto_Index", handleInvalid="keep")

assembler = VectorAssembler(
    inputCols=["Sucursal_Index", "Producto_Index", "Cantidad", "Precio_Unitario"],
    outputCol="features",
    handleInvalid="skip"
)

# Pipeline de preprocesamiento
prep_pipeline = Pipeline(stages=[sucursal_indexer, producto_indexer, assembler])
df_final = prep_pipeline.fit(df_model).transform(df_model)

# Persistencia en memoria para mejorar rendimiento
df_final.cache()

# D. Evidencia del DataFrame final
print("--- Evidencia del DataFrame Final (features y label) ---")
df_final.select("features", "label").show(5, truncate=False)

--- Evidencia del DataFrame Final (features y label) ---
+---------------------+-----+
|features             |label|
+---------------------+-----+
|[2.0,2.0,3.0,989.78] |0    |
|[2.0,1.0,4.0,563.57] |0    |
|[1.0,0.0,4.0,1180.5] |0    |
|[0.0,2.0,3.0,2194.99]|0    |
|[1.0,6.0,3.0,2470.44]|1    |
+---------------------+-----+
only showing top 5 rows


## 2. Entrenamiento de un modelo supervisado con MLlib
Uso de `StringIndexer` y `VectorAssembler` para conformar las características (features). Se excluyen intencionalmente las columnas `Monto_Total` y `Hora` del VectorAssembler para evitar Data Leakage, ya que fueron utilizadas para construir la variable objetivo. Se utiliza `Pipeline` para orquestar las transformaciones junto con el algoritmo `RandomForestClassifier`.

In [33]:
# A. División de datos en conjuntos de Entrenamiento y Prueba
train_data, test_data = df_final.randomSplit([0.8, 0.2], seed=42)
print(f"Registros de entrenamiento: {train_data.count()}")
print(f"Registros de prueba: {test_data.count()}")

# B. Configuración y Entrenamiento del modelo RandomForest
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxDepth=5, seed=42)
model = rf.fit(train_data)

# C. Predicciones sobre el set de pruebas
predictions = model.transform(test_data)

# D. Evidencia de la tabla final
print("\n--- Resultados de Predicción ---")
# Se muestran las columnas requeridas para validar el resultado del modelo
predictions.select("label", "prediction", "probability").show(10, truncate=False)

Registros de entrenamiento: 114
Registros de prueba: 32

--- Resultados de Predicción ---
+-----+----------+-----------------------------------------+
|label|prediction|probability                              |
+-----+----------+-----------------------------------------+
|0    |0.0       |[0.9775648474845019,0.022435152515498102]|
|0    |0.0       |[0.6968254168517097,0.30317458314829016] |
|0    |0.0       |[0.632288356784554,0.367711643215446]    |
|0    |0.0       |[0.997158401780555,0.002841598219445041] |
|1    |1.0       |[0.49918652796282087,0.5008134720371791] |
|1    |0.0       |[0.9910597432506743,0.008940256749325737]|
|0    |0.0       |[0.9767328698656614,0.023267130134338652]|
|0    |0.0       |[0.9424324185439334,0.05756758145606658] |
|0    |0.0       |[0.9957630529433456,0.004236947056654343]|
|0    |0.0       |[0.956761974194694,0.043238025805305905] |
+-----+----------+-----------------------------------------+
only showing top 10 rows


## 3. Evaluación del Modelo y Justificación Técnica
Cálculo de métricas de clasificación para validar el desempeño predictivo sobre datos no vistos, evitando el sobreajuste.

In [34]:
# A. Exactitud (Accuracy)
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator_acc.evaluate(predictions)

# B. Área bajo la curva ROC (AreaUnderROC)
evaluator_roc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
roc_auc = evaluator_roc.evaluate(predictions)

print("--- Métricas de Desempeño ---")
print(f"Exactitud (Accuracy): {accuracy:.4f}")
print(f"Área bajo la curva ROC (AUC): {roc_auc:.4f}")

--- Métricas de Desempeño ---
Exactitud (Accuracy): 0.9375
Área bajo la curva ROC (AUC): 0.6897


### Justificación Técnica y Estratégica del Modelo Predictivo

Se eligió **Random Forest** por su robustez ante ruido y su capacidad para procesar variables categóricas evitando el sobreajuste. La variable `label` identificó anomalías operativas (altos montos nocturnos). Para evitar la fuga de datos (*Data Leakage*), se excluyeron `Hora` y `Monto_Total` de las *features*, forzando al modelo a detectar patrones subyacentes.

**Análisis de Resultados:**
Se evaluó con Accuracy (0.9375) y AreaUnderROC (0.6897). Debido al fuerte desbalance de clases, el AUC es la métrica de control. Este resultado demuestra un rendimiento realista y matemáticamente válido que captura la tendencia predictiva sin memorizar resultados de forma determinista.

**Mejoras Futuras:**
1. **Validación Cruzada** con `ParamGridBuilder` para ajustar hiperparámetros (`maxDepth`, `numTrees`).
2. **Feature Engineering Avanzado** para incorporar variables externas (ej. clima local, fechas festivas) y cruzar atributos predictivos.